# Data Preparation

Match original panoramic radiographs with their XML annotations, split images into training/validation/test sets, and export a bounding-box CSV. Only original images are used to avoid augmentation leakage.

## Locate original radiographs and XML files

Match by **file stem** rather than the `<filename>` field inside XML, which may refer to a different image name.

In [1]:
from pathlib import Path
import xml.etree.ElementTree as ET
import pandas as pd
from sklearn.model_selection import train_test_split

#Set the dataset root relative to the notebooks folder
DATASET_DIR = Path("../dataset")

#Find original radiographs only; exclude augmented JPG images
images = sorted(
    f for f in DATASET_DIR.rglob("*.jpg")
    if f.parent.name == "Original JPG Images"
)

xml_files = {
    f.stem: f for f in DATASET_DIR.rglob("*.xml")
}

#Match each JPG to its XML file using the same filename stem
image_pairs = [
    (img, xml_files[img.stem])
    for img in images if img.stem in xml_files
]

print("Original images:", len(images))
print("Matched image pairs:", len(image_pairs))

Original images: 3875
Matched image pairs: 3875


## Split images (70% / 15% / 15%)

Split at the radiograph level **before** extracting lesion crops. A fixed random seed makes the results reproducible. If patient identifiers become available, a patient-level split would be preferable.

In [2]:
#First reserve 30% of radiographs for validation and testing
#Then divide the remaining 30% equally between both sets
train_pairs, remaining_pairs = train_test_split(
    image_pairs,
    test_size=0.30,
    random_state=42,
    shuffle=True
)

val_pairs, test_pairs = train_test_split(
    remaining_pairs,
    test_size=0.50,
    random_state=42,
    shuffle=True
)

print("Training images:", len(train_pairs))
print("Validation images:", len(val_pairs))
print("Test images:", len(test_pairs))
print("Total:", len(train_pairs) + len(val_pairs) + len(test_pairs))

Training images: 2712
Validation images: 581
Test images: 582
Total: 3875


## Export bounding-box annotations

Each CSV row represents **one lesion**, so multiple rows can belong to the same radiograph. The CSV will be saved to `../results/dataset_annotations.csv`.

In [3]:
#Read all XML bounding boxes and assign each lesion to its image split
records = []

datasets = {
    "train": train_pairs,
    "validation": val_pairs,
    "test": test_pairs
}

for split, pairs in datasets.items():

    for img_path, xml_path in pairs:

        root = ET.parse(xml_path).getroot()

        for obj in root.findall("object"):

            box = obj.find("bndbox")

            records.append({
                "image": str(img_path),
                "split": split,
                "class": int(obj.findtext("name")),
                "xmin": int(box.findtext("xmin")),
                "ymin": int(box.findtext("ymin")),
                "xmax": int(box.findtext("xmax")),
                "ymax": int(box.findtext("ymax"))
            })

#Save one row per lesion; keep the image path for later cropping
df = pd.DataFrame(records)

output_dir = Path("../results")
output_dir.mkdir(exist_ok=True)

df.to_csv(output_dir / "dataset_annotations.csv", index=False)

print("Total annotations:", len(df))
print("\nAnnotations by split:")
print(pd.crosstab(df["split"], df["class"]))

Total annotations: 6631

Annotations by split:
class          3     4    5
split                      
test         628   269  103
train       2904  1193  476
validation   666   274  118
